# P07 SkyOps - Week 4: Source-to-Bronze Ingestion

### ZENAIZ x BVRIT Hyderabad Data Engineering Internship

**Notebook:** `notebooks/02_bronze_ingestion.ipynb`
**Learning approach:** Spark SQL first
**Week 4 scope:** approved SkyOps batch source files to Bronze Delta tables

This notebook applies the Week 4 method (from the PageLoop worked example) to the
approved **P07 SkyOps** project files:

| Source file | Bronze target |
|---|---|
| `airports.csv` | `bronze_airports` |
| `carriers.csv` | `bronze_carriers` |
| `routes.csv` | `bronze_routes` |
| `flights.csv` | `bronze_flights` |

These four names match the Exact table/object register in the P07 SkyOps
Playbook (Part 2, section 11). Do not rename them.

## Week 4 boundary

Bronze preserves source business values **without cleaning**. No type casting,
uppercasing, deduplication, HHMM validation or DQ rule belongs in this notebook.
Those activities start in Week 5 (Silver Candidate) and later (DQ/Trusted Silver).

A reviewer must be able to take any Bronze row, trace it back to
`_source_file_name` / `_ingestion_run_id` / `_record_hash`, and see the
untouched original value - including invalid ones (e.g. a malformed HHMM time).

# Part 1 - Prepare Databricks

Before running this notebook:

1. import this notebook into Databricks;
2. attach Serverless (or cluster) compute;
3. upload the four approved SkyOps batch files to
   `%s`;
4. keep the filenames unchanged (`airports.csv`, `carriers.csv`, `routes.csv`,
   `flights.csv`);
5. run the notebook from the first code cell downward.

If your workspace uses a different approved catalog, schema or Volume, change
the paths and SQL context consistently before continuing.

## 1.1 Select the catalog and schema

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT
  current_catalog() AS active_catalog,
  current_schema() AS active_schema;

Expected observation: the result should show `workspace` and `default`. If it does not, stop and correct the approved catalog or schema before continuing.

## 1.2 Confirm the files in the Volume

In [ ]:
%fs
ls /Volumes/workspace/default/skyops

Confirm that the listing includes:

```text
airports.csv
carriers.csv
routes.csv
flights.csv
```

If one file is missing or has a different name, correct the upload before
running any ingestion cell.

# Part 2 - Understand the Bronze contract

Per the Playbook's Bronze metadata contract (Part 2, section 12), every Bronze
table must retain:

- `source_file_name`
- `source_record_key` (or a source row hash)
- `source_row_number`
- `ingestion_timestamp`
- `run_id` / `batch_id`
- `source_period`
- the raw payload / original columns, without silent cleaning

`flights.csv` already carries `source_record_key`, `source_period`,
`source_original_filename` and `source_row_number` as **approved business
columns** (the source file was built to that contract). Bronze must not
fabricate or overwrite them - it only adds ingestion-level lineage on top.

`airports.csv`, `carriers.csv` and `routes.csv` are small static reference
files with no such columns, so Bronze derives `_source_row_number` and
`_source_period` itself during ingestion.

Technical columns added by this notebook are prefixed with `_` so they are
never confused with an approved business field.

## 2.1 Define the controlled run values

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW week4_run_control AS
SELECT
  'W04_SKYOPS_RUN01' AS ingestion_run_id,
  'skyops_source_v1.0' AS schema_version;

In [ ]:
%sql
SELECT * FROM week4_run_control;

Keep `ingestion_run_id` and `schema_version` unchanged while building and
while performing the repeat-run test in Part 7. Change them only for a
genuinely new, mentor-approved batch run.

## 2.2 Why this notebook uses a controlled full refresh

The SkyOps Week 4 inputs are a fixed set of approved batch files. Each Bronze
table is written with `CREATE OR REPLACE TABLE`. When the same cell is rerun
against the same unchanged source, the table is refreshed - not appended - so
source and Bronze counts stay comparable and reruns do not duplicate rows.
This is a safe Week 4 pattern for static files; incremental ingestion is a
later design decision.

# Part 3 - Build `bronze_airports`

Airports is the reference-table worked example. Carriers and routes repeat the
same seven-step pattern in Parts 4 and 5.

## 3.1 Read and inspect `airports.csv`

Declare the schema explicitly, keep every field as `STRING`, and capture any
unparsable line in `_corrupt_record` rather than dropping it silently.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW airports_source
(
  airport_code STRING,
  airport_name STRING,
  city STRING,
  state_region STRING,
  active_flag STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/skyops/airports.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT * FROM airports_source LIMIT 10;

Expected observation: `airport_code`, `airport_name`, `city`, `state_region`, `active_flag` should all be visible and unmodified.

In [ ]:
%sql
DESCRIBE airports_source;

In [ ]:
%sql
SELECT COUNT(*) AS airports_source_count FROM airports_source;

Record the displayed count in your Week 4 log.

## 3.2 Add airports Bronze metadata

`monotonically_increasing_id()` gives a stable row ordinal for this
single, unpartitioned reference file, used as `_source_row_number`. The record
hash fingerprints the approved business columns in a fixed order so future
comparisons can detect any drift.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW airports_bronze_ready AS
SELECT
  s.airport_code,
  s.airport_name,
  s.city,
  s.state_region,
  s.active_flag,
  monotonically_increasing_id() + 1 AS _source_row_number,
  'airports.csv' AS _source_file_name,
  '/Volumes/workspace/default/skyops/airports.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  date_format(current_date(), 'yyyy-MM') AS _source_period,
  sha2(concat_ws('||',
    coalesce(cast(s.airport_code AS STRING), '<NULL>'),
    coalesce(cast(s.airport_name AS STRING), '<NULL>'),
    coalesce(cast(s.city AS STRING), '<NULL>'),
    coalesce(cast(s.state_region AS STRING), '<NULL>'),
    coalesce(cast(s.active_flag AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  cast(s._corrupt_record AS STRING) AS _rescued_payload
FROM airports_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
SELECT * FROM airports_bronze_ready LIMIT 10;

## 3.3 Create the `bronze_airports` Delta table

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_airports
USING DELTA
AS
SELECT *
FROM airports_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_airports;

In [ ]:
%sql
DESCRIBE TABLE bronze_airports;

In [ ]:
%sql
SELECT * FROM bronze_airports LIMIT 10;

## 3.4 Reconcile `airports`

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM airports_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_airports) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM airports_source)
       = (SELECT COUNT(*) FROM bronze_airports)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

Airports checkpoint:

- the source view opens and shows the approved five business columns;
- `bronze_airports` exists as a Delta table;
- technical metadata is populated;
- reconciliation shows `PASS`.

# Part 4 - Build `bronze_carriers`

Carriers follows the same method used for airports.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW carriers_source
(
  carrier_code STRING,
  carrier_name STRING,
  active_flag STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/skyops/carriers.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT * FROM carriers_source LIMIT 10;

In [ ]:
%sql
DESCRIBE carriers_source;

In [ ]:
%sql
SELECT COUNT(*) AS carriers_source_count FROM carriers_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW carriers_bronze_ready AS
SELECT
  s.carrier_code,
  s.carrier_name,
  s.active_flag,
  monotonically_increasing_id() + 1 AS _source_row_number,
  'carriers.csv' AS _source_file_name,
  '/Volumes/workspace/default/skyops/carriers.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  date_format(current_date(), 'yyyy-MM') AS _source_period,
  sha2(concat_ws('||',
    coalesce(cast(s.carrier_code AS STRING), '<NULL>'),
    coalesce(cast(s.carrier_name AS STRING), '<NULL>'),
    coalesce(cast(s.active_flag AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  cast(s._corrupt_record AS STRING) AS _rescued_payload
FROM carriers_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
SELECT * FROM carriers_bronze_ready LIMIT 10;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_carriers
USING DELTA
AS
SELECT *
FROM carriers_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_carriers;

In [ ]:
%sql
SELECT * FROM bronze_carriers LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM carriers_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_carriers) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM carriers_source)
       = (SELECT COUNT(*) FROM bronze_carriers)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

Carriers checkpoint:

- the source view shows `carrier_code`, `carrier_name`, `active_flag`;
- `bronze_carriers` exists as a Delta table;
- reconciliation shows `PASS`.

# Part 5 - Build `bronze_routes`

Routes has six business columns, including two numeric-looking fields
(`distance_miles`) which are still read and stored as `STRING` in Bronze -
casting to a numeric type is a Silver concern.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW routes_source
(
  route_id STRING,
  origin_airport_code STRING,
  destination_airport_code STRING,
  route_label STRING,
  distance_miles STRING,
  distance_band STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/skyops/routes.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT * FROM routes_source LIMIT 10;

In [ ]:
%sql
DESCRIBE routes_source;

In [ ]:
%sql
SELECT COUNT(*) AS routes_source_count FROM routes_source;

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW routes_bronze_ready AS
SELECT
  s.route_id,
  s.origin_airport_code,
  s.destination_airport_code,
  s.route_label,
  s.distance_miles,
  s.distance_band,
  monotonically_increasing_id() + 1 AS _source_row_number,
  'routes.csv' AS _source_file_name,
  '/Volumes/workspace/default/skyops/routes.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  date_format(current_date(), 'yyyy-MM') AS _source_period,
  sha2(concat_ws('||',
    coalesce(cast(s.route_id AS STRING), '<NULL>'),
    coalesce(cast(s.origin_airport_code AS STRING), '<NULL>'),
    coalesce(cast(s.destination_airport_code AS STRING), '<NULL>'),
    coalesce(cast(s.route_label AS STRING), '<NULL>'),
    coalesce(cast(s.distance_miles AS STRING), '<NULL>'),
    coalesce(cast(s.distance_band AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  cast(s._corrupt_record AS STRING) AS _rescued_payload
FROM routes_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
SELECT * FROM routes_bronze_ready LIMIT 10;

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_routes
USING DELTA
AS
SELECT *
FROM routes_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_routes;

In [ ]:
%sql
SELECT * FROM bronze_routes LIMIT 10;

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM routes_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_routes) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM routes_source)
       = (SELECT COUNT(*) FROM bronze_routes)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

Routes checkpoint:

- all six business columns are present and unmodified;
- `bronze_routes` exists as a Delta table;
- reconciliation shows `PASS`.

# Part 6 - Build `bronze_flights`

Flights is the fact source: one row per flight occurrence, already carrying
`source_record_key`, `source_period`, `source_original_filename` and
`source_row_number` as approved business columns.

WEEK SCENARIO reminder: a row can have an invalid or missing HHMM value
(e.g. a blank `actual_departure_hhmm` for a cancelled flight). Bronze must
retain it exactly as read so Silver and DQ can explain the failure later - do
not null-check, cast or repair it here.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_source
(
  source_record_key STRING,
  flight_date STRING,
  reporting_carrier STRING,
  flight_number STRING,
  tail_number STRING,
  origin_airport_code STRING,
  destination_airport_code STRING,
  scheduled_departure_hhmm STRING,
  actual_departure_hhmm STRING,
  departure_delay_signed_minutes STRING,
  departure_delay_minutes STRING,
  scheduled_arrival_hhmm STRING,
  actual_arrival_hhmm STRING,
  arrival_delay_signed_minutes STRING,
  arrival_delay_minutes STRING,
  cancelled_flag STRING,
  cancellation_code STRING,
  diverted_flag STRING,
  scheduled_elapsed_minutes STRING,
  actual_elapsed_minutes STRING,
  air_time_minutes STRING,
  taxi_out_minutes STRING,
  taxi_in_minutes STRING,
  distance_miles STRING,
  carrier_delay_minutes STRING,
  weather_delay_minutes STRING,
  nas_delay_minutes STRING,
  security_delay_minutes STRING,
  late_aircraft_delay_minutes STRING,
  source_period STRING,
  source_original_filename STRING,
  source_row_number STRING,
  _corrupt_record STRING
)
USING CSV
OPTIONS (
  path '/Volumes/workspace/default/skyops/flights.csv',
  header 'true',
  mode 'PERMISSIVE',
  columnNameOfCorruptRecord '_corrupt_record'
);

In [ ]:
%sql
SELECT * FROM flights_source LIMIT 10;

In [ ]:
%sql
DESCRIBE flights_source;

In [ ]:
%sql
SELECT COUNT(*) AS flights_source_count FROM flights_source;

Record this count from your own run. It should be in the hundred-thousands for the full SkyOps flights extract.

## 6.2 Add flights Bronze metadata

This view keeps **every** business column from `flights_source` untouched
(including the source's own `source_record_key`, `source_period`,
`source_row_number` and `source_original_filename`), and adds only the
ingestion-level lineage columns: `_source_file_name`, `_source_file_path`,
`_ingested_at`, `_ingestion_run_id`, `_schema_version` and `_record_hash`.

The hash covers every business column in a fixed order, so any future
byte-level drift between this file and a later reload is detectable.

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW flights_bronze_ready AS
SELECT
  s.* EXCEPT (_corrupt_record),
  'flights.csv' AS _source_file_name,
  '/Volumes/workspace/default/skyops/flights.csv' AS _source_file_path,
  current_timestamp() AS _ingested_at,
  r.ingestion_run_id AS _ingestion_run_id,
  r.schema_version AS _schema_version,
  sha2(concat_ws('||',
    coalesce(cast(s.source_record_key AS STRING), '<NULL>'),
    coalesce(cast(s.flight_date AS STRING), '<NULL>'),
    coalesce(cast(s.reporting_carrier AS STRING), '<NULL>'),
    coalesce(cast(s.flight_number AS STRING), '<NULL>'),
    coalesce(cast(s.tail_number AS STRING), '<NULL>'),
    coalesce(cast(s.origin_airport_code AS STRING), '<NULL>'),
    coalesce(cast(s.destination_airport_code AS STRING), '<NULL>'),
    coalesce(cast(s.scheduled_departure_hhmm AS STRING), '<NULL>'),
    coalesce(cast(s.actual_departure_hhmm AS STRING), '<NULL>'),
    coalesce(cast(s.departure_delay_signed_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.departure_delay_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.scheduled_arrival_hhmm AS STRING), '<NULL>'),
    coalesce(cast(s.actual_arrival_hhmm AS STRING), '<NULL>'),
    coalesce(cast(s.arrival_delay_signed_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.arrival_delay_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.cancelled_flag AS STRING), '<NULL>'),
    coalesce(cast(s.cancellation_code AS STRING), '<NULL>'),
    coalesce(cast(s.diverted_flag AS STRING), '<NULL>'),
    coalesce(cast(s.scheduled_elapsed_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.actual_elapsed_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.air_time_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.taxi_out_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.taxi_in_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.distance_miles AS STRING), '<NULL>'),
    coalesce(cast(s.carrier_delay_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.weather_delay_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.nas_delay_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.security_delay_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.late_aircraft_delay_minutes AS STRING), '<NULL>'),
    coalesce(cast(s.source_period AS STRING), '<NULL>'),
    coalesce(cast(s.source_original_filename AS STRING), '<NULL>'),
    coalesce(cast(s.source_row_number AS STRING), '<NULL>')
  ), 256) AS _record_hash,
  cast(s._corrupt_record AS STRING) AS _rescued_payload
FROM flights_source s
CROSS JOIN week4_run_control r;

In [ ]:
%sql
SELECT
  source_record_key,
  flight_date,
  reporting_carrier,
  scheduled_departure_hhmm,
  actual_departure_hhmm,
  _source_file_name,
  _ingestion_run_id,
  _record_hash,
  _rescued_payload
FROM flights_bronze_ready
LIMIT 10;

## 6.3 Create the `bronze_flights` Delta table

In [ ]:
%sql
CREATE OR REPLACE TABLE bronze_flights
USING DELTA
AS
SELECT *
FROM flights_bronze_ready;

In [ ]:
%sql
DESCRIBE DETAIL bronze_flights;

In [ ]:
%sql
DESCRIBE TABLE bronze_flights;

In [ ]:
%sql
SELECT * FROM bronze_flights LIMIT 10;

## 6.4 Reconcile `flights`

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM flights_source) AS source_count,
  (SELECT COUNT(*) FROM bronze_flights) AS bronze_count,
  CASE
    WHEN (SELECT COUNT(*) FROM flights_source)
       = (SELECT COUNT(*) FROM bronze_flights)
    THEN 'PASS'
    ELSE 'INVESTIGATE'
  END AS reconciliation_status;

Flights checkpoint:

- all 31 approved business columns are present and unmodified, including the
  source's own `source_record_key` / `source_period` / `source_row_number`;
- `bronze_flights` exists as a Delta table;
- ingestion metadata (`_source_file_name`, `_ingested_at`, `_ingestion_run_id`,
  `_record_hash`) is populated;
- reconciliation shows `PASS`.

# Part 7 - Validate the complete Bronze load

Four persistent Bronze Delta tables now exist. This part validates them as one
controlled Week 4 load, exactly as required by the Exit checklist in Part 2 of
the Playbook.

## 7.1 Confirm that all four tables exist

In [ ]:
%sql
SHOW TABLES LIKE 'bronze_*';

Expected observation: `bronze_airports`, `bronze_carriers`, `bronze_routes` and `bronze_flights` should all be listed.

## 7.2 One reconciliation summary for all four sources

In [ ]:
%sql
WITH counts AS (
  SELECT 'airports' AS dataset,
         (SELECT COUNT(*) FROM airports_source) AS source_count,
         (SELECT COUNT(*) FROM bronze_airports) AS bronze_count
  UNION ALL
  SELECT 'carriers',
         (SELECT COUNT(*) FROM carriers_source),
         (SELECT COUNT(*) FROM bronze_carriers)
  UNION ALL
  SELECT 'routes',
         (SELECT COUNT(*) FROM routes_source),
         (SELECT COUNT(*) FROM bronze_routes)
  UNION ALL
  SELECT 'flights',
         (SELECT COUNT(*) FROM flights_source),
         (SELECT COUNT(*) FROM bronze_flights)
)
SELECT
  *,
  bronze_count - source_count AS count_difference,
  CASE WHEN source_count = bronze_count THEN 'PASS'
       ELSE 'INVESTIGATE' END AS status
FROM counts;

All four rows must show `PASS` before Week 4 can close. A difference does not automatically mean the source is wrong - stop, inspect path, reader, write method and table state, and explain the cause in the Week Log.

## 7.3 Check metadata completeness

In [ ]:
%sql
SELECT 'airports' AS dataset,
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END) AS missing_file,
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END) AS missing_time,
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END) AS missing_run,
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END) AS missing_hash
FROM bronze_airports
UNION ALL
SELECT 'carriers',
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_carriers
UNION ALL
SELECT 'routes',
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_routes
UNION ALL
SELECT 'flights',
       SUM(CASE WHEN _source_file_name IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingested_at IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _ingestion_run_id IS NULL THEN 1 ELSE 0 END),
       SUM(CASE WHEN _record_hash IS NULL THEN 1 ELSE 0 END)
FROM bronze_flights;

Every value should be zero. Investigate any non-zero result before capturing evidence.

## 7.4 Check parser / rescued-record context

In [ ]:
%sql
SELECT 'airports' AS dataset, COUNT(*) AS rows_with_rescued_context
FROM bronze_airports WHERE _rescued_payload IS NOT NULL
UNION ALL
SELECT 'carriers', COUNT(*) FROM bronze_carriers WHERE _rescued_payload IS NOT NULL
UNION ALL
SELECT 'routes', COUNT(*) FROM bronze_routes WHERE _rescued_payload IS NOT NULL
UNION ALL
SELECT 'flights', COUNT(*) FROM bronze_flights WHERE _rescued_payload IS NOT NULL;

A non-zero result is not automatically a failure - it means the reader captured parser context for that row. Preserve it in Bronze; routing or correcting it is a later-week (Silver/DQ) activity.

## 7.5 Repeat-run test

Record the current count summary above. Then rerun the four
`CREATE OR REPLACE TABLE` cells in Parts 3-6 without changing the source files
or `week4_run_control` values. Finally, rerun the next query.

In [ ]:
%sql
SELECT
  (SELECT COUNT(*) FROM bronze_airports) AS airports_after_rerun,
  (SELECT COUNT(*) FROM bronze_carriers) AS carriers_after_rerun,
  (SELECT COUNT(*) FROM bronze_routes) AS routes_after_rerun,
  (SELECT COUNT(*) FROM bronze_flights) AS flights_after_rerun;

Compare with the before-rerun counts. They must be unchanged - `CREATE OR REPLACE TABLE` performs a controlled full refresh, not an append, so rerunning the same batch cannot silently double the rows.

## 7.6 Inspect Delta history

In [ ]:
%sql
DESCRIBE HISTORY bronze_airports;

In [ ]:
%sql
DESCRIBE HISTORY bronze_carriers;

In [ ]:
%sql
DESCRIBE HISTORY bronze_routes;

In [ ]:
%sql
DESCRIBE HISTORY bronze_flights;

Each history should show a `CREATE OR REPLACE TABLE AS SELECT` operation per run, with no unexplained extra versions.

# Part 8 - Evidence and Week 4 exit checklist

## Evidence to capture

| Evidence ID | What to capture |
|---|---|
| W04-E01 | Volume listing with all four approved source files |
| W04-E02 | `bronze_airports` schema/details |
| W04-E03 | `bronze_carriers` schema/details |
| W04-E04 | `bronze_routes` schema/details |
| W04-E05 | `bronze_flights` schema/details |
| W04-E06 | Four-source reconciliation query with `PASS` status |
| W04-E07 | Metadata-completeness query (all zero) |
| W04-E08 | Repeat-run before/after counts |
| W04-E09 | `DESCRIBE HISTORY` for at least one Bronze table |
| W04-E10 | Catalog Explorer table list |

## Repository updates

- save this notebook as `notebooks/02_bronze_ingestion.ipynb`;
- update the source inventory / data dictionary;
- add Week 4 evidence references in `screenshots/README.md`;
- complete `weekly_logs/week04_log.md` (run IDs, results, AI note);
- complete `docs/pipeline_walkthrough.md` with the initial layer description;
- commit only approved GitHub-safe files - never the full restricted dataset,
  tokens, or workspace credentials.

## Exit checklist

- [ ] `bronze_flights`, `bronze_airports`, `bronze_carriers`, `bronze_routes` all exist and reconcile.
- [ ] Metadata (`_source_file_name`, `_ingested_at`, `_ingestion_run_id`, `_record_hash`, plus the source's own `source_record_key`/`source_period`/`source_row_number` for flights) is complete.
- [ ] No source business field was cleaned, cast or corrected in Bronze.
- [ ] Rerun behaviour is demonstrated and documented as controlled (not duplicating).
- [ ] Evidence and Week Log are captured.

VIVA QUESTION reminder: be ready to explain why Bronze must preserve an
invalid source value (e.g. a malformed HHMM) instead of fixing it immediately.